# updatesupport downstream of DoWhy

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nahuaque/updatesupport/blob/main/examples/notebooks/dowhy_downstream_reporting_colab.ipynb)

This notebook shows how `updatesupport` sits downstream of a [DoWhy](https://www.pywhy.org/dowhy/) causal workflow.

The workflow is:

1. Use DoWhy to state and estimate a causal effect.
2. Build a row-level effect surface, `tau_hat`, for reporting.
3. Use `updatesupport` to ask whether a coarse public report is stable under hidden-composition shifts.

`updatesupport` is not replacing causal identification or estimation. DoWhy owns the causal estimand and estimate. `updatesupport` audits the reporting layer: if we publish effects by coarse public buckets, how much could the aggregate reported effect move if hidden composition inside those buckets changed?

## 1. Install and import

Run the install cell once in Colab. It installs DoWhy, `updatesupport`, and plotting dependencies. This notebook deliberately avoids scikit-learn imports.

In [ ]:
%pip install -q --upgrade "updatesupport[dowhy,examples]>=0.1.2" "dowhy>=0.13,<0.15" "pandas>=2.2,<2.4" "seaborn>=0.13,<0.14" "matplotlib>=3.9,<3.11"

In [ ]:
from dataclasses import dataclass
from importlib.metadata import PackageNotFoundError, version

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import updatesupport as us

try:
    from dowhy import CausalModel

    DOWHY_AVAILABLE = True
    DOWHY_IMPORT_ERROR = None
except Exception as exc:
    CausalModel = None
    DOWHY_AVAILABLE = False
    DOWHY_IMPORT_ERROR = repr(exc)


def installed_version(package: str) -> str:
    try:
        return version(package)
    except PackageNotFoundError:
        return "not installed"


@dataclass(frozen=True)
class ScalarEstimate:
    """Small estimate object matching DoWhy's numeric .value convention."""

    value: float
    method_name: str = "weighted outcome-regression fallback"


print(f"updatesupport {installed_version('updatesupport')}")
print(f"dowhy {installed_version('dowhy')} (importable: {DOWHY_AVAILABLE})")
print(f"numpy {installed_version('numpy')}")
print(f"pandas {installed_version('pandas')}")
if not DOWHY_AVAILABLE:
    print(
        "DoWhy did not import in this runtime; continuing with a ScalarEstimate object."
    )
    print(f"DoWhy import error: {DOWHY_IMPORT_ERROR}")

sns.set_theme(
    style="whitegrid",
    context="notebook",
    palette="colorblind",
    rc={"axes.spines.right": False, "axes.spines.top": False},
)

## 2. Build a synthetic causal dataset

The data-generating process has observed confounding and a heterogeneous treatment effect, `true_tau`. DoWhy estimates the causal effect. The support audit below uses `tau_hat`, a row-level effect surface built from a transparent weighted outcome regression.

The public report will only use `age_band`, `region`, and `sex`. The hidden state also includes channel, risk band, severity band, and prior-use band.

In [ ]:
RNG = np.random.default_rng(42)
N = 1200

age = RNG.normal(44, 13, N).clip(18, 78)
income = RNG.lognormal(mean=10.45, sigma=0.45, size=N)
severity = RNG.beta(2.2, 4.0, N)
prior_use = RNG.poisson(1.2 + 2.6 * severity, N)
risk_score = (0.55 * severity + 0.018 * prior_use + RNG.normal(0, 0.08, N)).clip(0, 1)

region = RNG.choice(["north", "south", "west"], size=N, p=[0.36, 0.34, 0.30])
sex = RNG.choice(["F", "M"], size=N, p=[0.52, 0.48])
channel = RNG.choice(["direct", "partner", "online"], size=N, p=[0.40, 0.28, 0.32])

age_band = pd.cut(
    age,
    bins=[17, 34, 49, 64, 90],
    labels=["18_34", "35_49", "50_64", "65_plus"],
).astype(str)
risk_band = pd.cut(
    risk_score,
    bins=[-0.01, 0.25, 0.45, 1.01],
    labels=["low", "medium", "high"],
).astype(str)
severity_band = pd.cut(
    severity,
    bins=[-0.01, 0.25, 0.50, 1.01],
    labels=["low", "medium", "high"],
).astype(str)
prior_use_band = pd.cut(
    prior_use,
    bins=[-1, 1, 4, 99],
    labels=["0_1", "2_4", "5_plus"],
).astype(str)

region_effect = (
    pd.Series(region).map({"north": 0.04, "south": -0.02, "west": 0.01}).to_numpy()
)
channel_effect = (
    pd.Series(channel)
    .map({"direct": 0.02, "partner": -0.03, "online": 0.05})
    .to_numpy()
)
true_tau = 0.18 + 0.20 * severity + 0.08 * risk_score + region_effect + channel_effect

propensity_logit = (
    -0.35
    + 1.1 * severity
    + 0.40 * (region == "south")
    + 0.35 * (channel == "online")
    - 0.25 * (sex == "F")
)
propensity = 1 / (1 + np.exp(-propensity_logit))
treatment = RNG.binomial(1, propensity)

baseline = (
    0.30
    + 0.006 * (age - 44)
    + 0.0000018 * (income - income.mean())
    + 0.35 * severity
    + 0.08 * (region == "south")
    - 0.04 * (channel == "direct")
)
outcome = baseline + true_tau * treatment + RNG.normal(0, 0.18, N)
sample_weight = RNG.uniform(0.7, 1.6, N)

df = pd.DataFrame(
    {
        "age": age,
        "income": income,
        "severity": severity,
        "prior_use": prior_use,
        "risk_score": risk_score,
        "region": region,
        "sex": sex,
        "channel": channel,
        "age_band": age_band,
        "risk_band": risk_band,
        "severity_band": severity_band,
        "prior_use_band": prior_use_band,
        "treatment": treatment,
        "outcome": outcome,
        "true_tau": true_tau,
        "sample_weight": sample_weight,
    }
)

PUBLIC_COLUMNS = ("age_band", "region", "sex")
HIDDEN_COLUMNS = (
    "age_band",
    "region",
    "sex",
    "channel",
    "risk_band",
    "severity_band",
    "prior_use_band",
)
CANDIDATE_REFINEMENTS = ("channel", "risk_band", "severity_band", "prior_use_band")
COMMON_CAUSES = [
    "age",
    "income",
    "severity",
    "prior_use",
    "risk_score",
    "region",
    "sex",
    "channel",
]

df.head()

## 3. Estimate the causal effect with DoWhy

DoWhy states the causal model, identifies a backdoor estimand, and estimates the average treatment effect with linear regression adjustment.

If DoWhy cannot import in a managed notebook runtime, this cell keeps going with a scalar weighted difference estimate. The support audit still demonstrates the downstream contract, but the preferred path is the DoWhy branch.

In [ ]:
if DOWHY_AVAILABLE:
    causal_model = CausalModel(
        data=df,
        treatment="treatment",
        outcome="outcome",
        common_causes=COMMON_CAUSES,
    )
    identified_estimand = causal_model.identify_effect(proceed_when_unidentifiable=True)
    estimate = causal_model.estimate_effect(
        identified_estimand,
        method_name="backdoor.linear_regression",
        target_units="ate",
    )
    effect_value = float(estimate.value)
    print("DoWhy estimate")
    print(estimate)
else:
    treated = df["treatment"].to_numpy().astype(bool)
    treated_mean = np.average(
        df.loc[treated, "outcome"], weights=df.loc[treated, "sample_weight"]
    )
    control_mean = np.average(
        df.loc[~treated, "outcome"], weights=df.loc[~treated, "sample_weight"]
    )
    effect_value = float(treated_mean - control_mean)
    estimate = ScalarEstimate(effect_value)
    print(f"Fallback scalar estimate: {effect_value:.4f}")

print(f"scalar effect used as DoWhy baseline: {effect_value:.4f}")
print(
    f"mean true effect in synthetic data: {np.average(df['true_tau'], weights=df['sample_weight']):.4f}"
)

## 4. Build `tau_hat` for reporting

DoWhy's standard linear-regression estimator returns an average effect. For a representation-stability audit, we also want a row-level effect surface so hidden cells can differ.

Here we fit a transparent weighted linear outcome model with treatment interactions. This is deliberately simple and NumPy-only. In a production causal workflow, this column could come from any upstream estimator, including DoWhy extensions, DoubleML, uplift models, causal forests, or a reviewed internal model.

In [ ]:
def standardized(values):
    values = np.asarray(values, dtype=float)
    return (values - values.mean()) / values.std()


def outcome_design(frame: pd.DataFrame) -> pd.DataFrame:
    age_z = standardized(frame["age"])
    income_z = standardized(np.log(frame["income"]))
    prior_z = standardized(frame["prior_use"])
    treatment = frame["treatment"].astype(float).to_numpy()
    design = pd.DataFrame(
        {
            "intercept": 1.0,
            "treatment": treatment,
            "age_z": age_z,
            "income_z": income_z,
            "severity": frame["severity"].to_numpy(),
            "risk_score": frame["risk_score"].to_numpy(),
            "prior_use_z": prior_z,
            "region_south": (frame["region"] == "south").astype(float).to_numpy(),
            "region_west": (frame["region"] == "west").astype(float).to_numpy(),
            "sex_m": (frame["sex"] == "M").astype(float).to_numpy(),
            "channel_online": (frame["channel"] == "online").astype(float).to_numpy(),
            "channel_partner": (frame["channel"] == "partner").astype(float).to_numpy(),
        }
    )
    design["treat_x_age_z"] = treatment * design["age_z"]
    design["treat_x_severity"] = treatment * design["severity"]
    design["treat_x_risk_score"] = treatment * design["risk_score"]
    design["treat_x_prior_use_z"] = treatment * design["prior_use_z"]
    design["treat_x_online"] = treatment * design["channel_online"]
    design["treat_x_partner"] = treatment * design["channel_partner"]
    return design


def fit_weighted_lstsq(design: pd.DataFrame, y, weights) -> pd.Series:
    root_w = np.sqrt(np.asarray(weights, dtype=float))
    xw = design.to_numpy(dtype=float) * root_w[:, None]
    yw = np.asarray(y, dtype=float) * root_w
    coef, *_ = np.linalg.lstsq(xw, yw, rcond=None)
    return pd.Series(coef, index=design.columns)


def treatment_effect_surface(
    frame: pd.DataFrame, coefficients: pd.Series
) -> np.ndarray:
    age_z = standardized(frame["age"])
    prior_z = standardized(frame["prior_use"])
    online = (frame["channel"] == "online").astype(float).to_numpy()
    partner = (frame["channel"] == "partner").astype(float).to_numpy()
    return (
        coefficients["treatment"]
        + coefficients["treat_x_age_z"] * age_z
        + coefficients["treat_x_severity"] * frame["severity"].to_numpy()
        + coefficients["treat_x_risk_score"] * frame["risk_score"].to_numpy()
        + coefficients["treat_x_prior_use_z"] * prior_z
        + coefficients["treat_x_online"] * online
        + coefficients["treat_x_partner"] * partner
    )


design = outcome_design(df)
coefficients = fit_weighted_lstsq(design, df["outcome"], df["sample_weight"])
tau_hat = treatment_effect_surface(df, coefficients)

adapted = us.adapt_dowhy_effects(
    estimate,
    df,
    effect_values=tau_hat,
    effect_column="tau_hat",
)
effect_df = pd.DataFrame(adapted.rows)

print(f"adapter source: {adapted.source}")
print(f"effect column: {adapted.effect_column}")
effect_df[
    ["true_tau", "tau_hat", "age_band", "region", "sex", "channel", "risk_band"]
].head()

## 5. Inspect the effect surface

The left chart checks whether the row-level effect surface tracks the synthetic ground truth. The right chart shows how `tau_hat` varies across hidden fields. Hidden variation is what can make a coarse public report unstable.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)

sns.scatterplot(
    data=effect_df.sample(450, random_state=1),
    x="true_tau",
    y="tau_hat",
    hue="risk_band",
    alpha=0.75,
    ax=axes[0],
)
lo = min(effect_df["true_tau"].min(), effect_df["tau_hat"].min())
hi = max(effect_df["true_tau"].max(), effect_df["tau_hat"].max())
axes[0].plot([lo, hi], [lo, hi], color="#C44E52", linestyle="--", linewidth=1.5)
axes[0].set_title("Row-level effect surface vs synthetic truth")
axes[0].set_xlabel("true_tau")
axes[0].set_ylabel("tau_hat")

sns.boxplot(data=effect_df, x="risk_band", y="tau_hat", hue="channel", ax=axes[1])
axes[1].set_title("Estimated effect heterogeneity in hidden fields")
axes[1].set_xlabel("Risk band")
axes[1].set_ylabel("tau_hat")
plt.show()

## 6. Audit the public causal report

Now the causal estimate and row-level effect surface exist. The audit asks a reporting question:

> If we publish effects only by `age_band x region x sex`, could the aggregate reported `tau_hat` change if the hidden mix of `channel`, `risk_band`, `severity_band`, and `prior_use_band` changed inside those public buckets?

The hidden-composition interval is not a confidence interval. It is a support-stability interval under the chosen hidden-mix stress test.

In [ ]:
audit = us.audit_dowhy_effects(
    effect_df,
    estimate=estimate,
    public=PUBLIC_COLUMNS,
    hidden=HIDDEN_COLUMNS,
    effect="tau_hat",
    weight="sample_weight",
    candidate_refinements=CANDIDATE_REFINEMENTS,
    q=us.q_bounded_shift(0.45),
    min_cell_weight=4,
    top=8,
    title="DoWhy Downstream Reporting Stability Audit",
)
report = audit.report
tables = report.to_tables()
summary = pd.DataFrame(tables["summary"]).iloc[0]

print(f"DoWhy baseline scalar effect: {audit.estimated_effect:.4f}")
print(f"observed aggregate tau_hat: {summary['observed_value']:.4f}")
print(f"hidden-composition interval: [{summary['lower']:.4f}, {summary['upper']:.4f}]")
print(f"ambiguity width: {summary['ambiguity']:.4f}")
print(f"public adequate: {summary['public_adequate']}")

## 7. Explain where ambiguity comes from

The first plot shows which public buckets contribute most to hidden-composition ambiguity. The second plot ranks candidate hidden variables by how much ambiguity would fall if we promoted that variable into the public report.

In [ ]:
fibers = pd.DataFrame(tables.get("worst_fibers", ()))
refinements = pd.DataFrame(tables.get("refinements", ()))

fig, axes = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)

if not fibers.empty:
    fibers = fibers.assign(
        public_label=fibers["public_value"].map(
            lambda value: " | ".join(map(str, value))
        )
    )
    sns.barplot(
        data=fibers.sort_values("contribution", ascending=False).head(8),
        y="public_label",
        x="contribution",
        color="#4C72B0",
        ax=axes[0],
    )
    axes[0].set_title("Public buckets driving ambiguity")
    axes[0].set_xlabel("Contribution to ambiguity")
    axes[0].set_ylabel("")
else:
    axes[0].text(0.5, 0.5, "No ambiguous public fibers", ha="center", va="center")
    axes[0].set_axis_off()

if not refinements.empty:
    sns.barplot(
        data=refinements.sort_values("reduction", ascending=False),
        y="column",
        x="reduction",
        color="#55A868",
        ax=axes[1],
    )
    axes[1].set_title("Best hidden-field refinements")
    axes[1].set_xlabel("Ambiguity reduction")
    axes[1].set_ylabel("")
else:
    axes[1].text(0.5, 0.5, "No refinement table", ha="center", va="center")
    axes[1].set_axis_off()

plt.show()

## 8. Read the report artifact

The Markdown report is the attachable artifact. It separates the causal estimate, hidden-composition ambiguity, public refinement recommendations, and limitations.

In [ ]:
markdown = audit.to_markdown()
print("\n".join(markdown.splitlines()[:90]))

## 9. Optional: package as a DoWhy-style refutation

When DoWhy is importable, `updatesupport` can package the hidden-composition interval as a DoWhy `CausalRefutation`-style artifact. The `new_effect` is an interval, not a second point estimate.

In [ ]:
if DOWHY_AVAILABLE:
    refutation = audit.to_refutation()
    print(refutation)
    print(f"updatesupport interval: {refutation.updatesupport_interval}")
    print(f"updatesupport ambiguity: {refutation.updatesupport_ambiguity:.4f}")
else:
    print("Skipping CausalRefutation conversion because DoWhy is not importable.")

## 10. Search for a smaller stable public representation

The frontier search asks which hidden fields are worth promoting into the public report. The goal is not to publish every internal covariate. The goal is to find a public representation that is small enough to communicate and stable enough for review.

In [ ]:
frontier = us.public_representation_frontier(
    adapted.rows,
    base_public=PUBLIC_COLUMNS,
    hidden=HIDDEN_COLUMNS,
    target="tau_hat",
    weight="sample_weight",
    candidate_refinements=CANDIDATE_REFINEMENTS,
    q_presets=("saturated", us.q_bounded_shift(0.45), "observed"),
    min_cell_weights=(4,),
    ambiguity_limit=0.020,
    bucket_budget=90,
    search="beam",
    beam_width=8,
    max_added_columns=3,
    max_evaluations=96,
    title="DoWhy Downstream Public Representation Frontier",
)

rows = [
    {
        "label": candidate.label,
        "public_cells": candidate.public_cells,
        "max_ambiguity": candidate.max_ambiguity,
        "added_columns": candidate.added_column_count,
        "stable": bool(candidate.passes_ambiguity_limit),
        "frontier": candidate in frontier.frontier,
    }
    for candidate in frontier.candidates
]
frame = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(11, 6))
sns.scatterplot(
    data=frame,
    x="public_cells",
    y="max_ambiguity",
    hue="stable",
    style="frontier",
    size="added_columns",
    sizes=(80, 320),
    ax=ax,
)
ax.axhline(0.020, color="#C44E52", linestyle="--", linewidth=1.5, label="review limit")
ax.set_title("Public representation frontier for tau_hat")
ax.set_xlabel("Public cells")
ax.set_ylabel("Worst ambiguity")
plt.show()

if frontier.minimal_stable is not None:
    print(f"minimal stable representation: {frontier.minimal_stable.label}")
else:
    print("no evaluated representation meets the ambiguity limit")

## 11. Takeaway

The handoff is:

```python
estimate = causal_model.estimate_effect(...)
adapted = us.adapt_dowhy_effects(estimate, df, effect_values=tau_hat)
audit = us.audit_dowhy_effects(adapted.rows, estimate=estimate, ...)
```

DoWhy handles the causal estimand and scalar causal estimate. `updatesupport` asks whether the public representation used to report a supplied effect surface is stable under hidden-composition shifts.